# Revisi - Tahap 4: Cek Kualitas Data (Metode Statistik + Semantik)

Metode **KETIGA**, pelengkap dua metode sebelumnya. Ketiganya saling lengkapi:

| metode | file | menjawab |
|---|---|---|
| heuristik aturan | `src.preprop.inspect_data` | konsistensi internal & kerusakan format |
| LLM-as-judge | `src.eval.judge_data_quality` | kebenaran matematis gold/solusi |
| **statistik + semantik (ini)** | `src.eval.data_quality_stats` | **sebaran label & kemiripan teks** |

Semua **deterministik** (CPU, tanpa LLM, bisa diulang persis). Kemiripan teks pakai
TF-IDF char n-gram + cosine (`char_wb`, 3-5 gram) -> tak perlu unduh model, hasil stabil.

Yang baru di sini (tak dicek dua metode lain):
- **kontaminasi train<->test** (`bocor_train_rate`): kalau soal test mirip soal train,
  skor eval (pass@k) jadi optimistik palsu. ~0 -> bukti dekontaminasi berhasil.
- **degenerasi label** (`entropi_jawaban`, `top_jawaban_share`): apakah jawaban gold
  nyebar sehat atau didominasi satu nilai (bisa "ditebak" -> eval tak bermakna).
- **bahasa** (`lang_id_rate`): apakah soal benar Bahasa Indonesia.

In [1]:
import os, sys
p = os.getcwd()
while not os.path.isdir(os.path.join(p, 'src')) and os.path.dirname(p) != p:
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
import pandas as pd
from IPython.display import Markdown, display
print('repo root:', p)

repo root: D:\Main Storage\Vscode\FP_NLP\FP_NLP


In [2]:
TEST = {
    'numglue_test': 'data/sft/test/numglue_test.jsonl',
    'easy_test':    'data/sft/test/easy_test.jsonl',
}
TRAIN = {
    'cot':   'data/sft/train/cot.jsonl',
    'nocot': 'data/sft/train/nocot.jsonl',
}

## Apa yang dicek & dinilai (metode statistik + semantik)

| metrik | yang dicek | baik bila |
|---|---|---|
| `lang_id_rate` | soal terdeteksi Bahasa Indonesia | tinggi |
| `dup_semantik_rate` | soal near-duplikat soal lain di set sama | rendah |
| `bocor_train_rate` | (test) soal test mirip soal train (kontaminasi) | **rendah** |
| `entropi_jawaban` | sebaran nilai jawaban (0..1) | tinggi (nyebar) |
| `top_jawaban_share` | porsi nilai jawaban paling sering | rendah |
| `len_outlier_rate` | panjang soal outlier (robust z) -> kepotong/sampah | rendah |

## Hasil

`bocor_train_rate` dihitung dengan acuan gabungan soal `cot`+`nocot` (pool train).

In [3]:
from src.eval.data_quality_stats import stats_metrics, _soals
from src.preprop.inspect_data import _read_jsonl, _kind
# pool train sbg acuan kebocoran
train_pool = []
for pth in TRAIN.values():
    rows = _read_jsonl(pth)
    train_pool += _soals(rows, _kind(rows))
print('pool train soal:', len(train_pool))
gold = pd.DataFrame([stats_metrics(p, train_pool) for p in TEST.values()]).set_index('dataset')
train = pd.DataFrame([stats_metrics(p, train_pool) for p in TRAIN.values()]).set_index('dataset')
print('TEST (gold):'); display(gold)
print('TRAIN (chatml):'); display(train)

pool train soal: 5418


TEST (gold):


,kind,n,lang_id_rate,dup_semantik_rate,entropi_jawaban,top_jawaban_share,len_outlier_rate,bocor_train_rate
dataset,,,,,,,,
numglue_test,gold,300,0.993,0.007,0.906,0.077,0.057,0.073
easy_test,gold,300,0.889,0.000,0.969,0.030,0.033,0.010


TRAIN (chatml):


,kind,n,lang_id_rate,dup_semantik_rate,entropi_jawaban,top_jawaban_share,len_outlier_rate
dataset,,,,,,,
cot,chatml,2709,0.929,0.049,0.918,0.027,0.024
nocot,chatml,2709,0.929,0.049,0.918,0.027,0.024


## Bukti 1 - Kontaminasi train<->test (soal test yang mirip soal train)

Kalau soal-soal ini benar bocor, model bisa "hafal" jawabannya dari training ->
skor eval untuk baris itu tak sah. Tampilkan pasangan cosine tertinggi.

In [4]:
from src.eval.data_quality_stats import leak_examples
for name, pth in TEST.items():
    rows = _read_jsonl(pth); soals = _soals(rows, _kind(rows))
    ex = leak_examples(soals, train_pool, thr=0.85, k=3)
    print('='*70); print(f'{name}: {len(ex)} contoh bocor teratas (thr=0.85)')
    for cos, te, tr in ex:
        print(f'  cosine={cos}')
        print(f'    TEST : {te[:110]}')
        print(f'    TRAIN: {tr[:110]}')

numglue_test: 3 contoh bocor teratas (thr=0.85)
  cosine=0.999
    TEST : Lakers mencetak total 80 poin dalam pertandingan basket melawan Bulls. Lakers membuat total 37 keranjang, yang
    TRAIN: Lakers mencetak total 80 poin dalam pertandingan basket melawan Bulls. Lakers membuat total 37 keranjang, yang
  cosine=0.992
    TEST : Seorang penjaga toko memiliki 6 deck kartu lengkap dan 7 kartu tambahan.
    TRAIN: Seorang penjaga toko memiliki 4 deck kartu lengkap dan 5 kartu tambahan.
  cosine=0.987
    TEST : Dua pengendara sepeda mulai dari titik yang sama dan bergerak ke arah yang berlawanan. Satu pengendara sepeda 
    TRAIN: Dua pengendara sepeda mulai dari titik yang sama dan bergerak ke arah yang berlawanan. Satu pengendara sepeda 


easy_test: 3 contoh bocor teratas (thr=0.85)
  cosine=0.893
    TEST : Diketahui $\vec{a} = \begin{pmatrix} 2 \\ -1 \\ 3 \end{pmatrix}$ dan $\vec{b} = \begin{pmatrix} 0 \\ 3 \\ -3 \
    TRAIN: Diketahui vektor $\vec{a} = \begin{pmatrix} 1 \\ 1 \\ 0 \end{pmatrix}$ dan vektor $\vec{b} = \begin{pmatrix} 1
  cosine=0.891
    TEST : Sebuah bola dijatuhkan ke lantai dari suatu tempat dengan ketinggian 1 meter. Setiap bola memantul, bola terse
    TRAIN: Sebuah bola tenis dijatuhkan ke lantai dari ketinggian 1 m. Setiap kali bola tersebut memantul, ketinggiannya 
  cosine=0.886
    TEST : Matriks X berordo 2x2 yang memenuhi persamaan $X \begin{pmatrix} 2 & 1 \ 3 & -5 \end{pmatrix} = \begin{pmatrix
    TRAIN: Tentukan hasil dari $\begin{pmatrix} 4 & 1 \ 9 & 2 \end{pmatrix} + \begin{pmatrix} 4 & 8 \ 9 & 9 \end{pmatrix}


## Bukti 2 - Near-duplikat di dalam train (parafrase yang lolos dedup persis)

`dup_semantik` nangkep soal yang nyaris sama tapi tak identik (beda angka/kata) -> redundansi
training. Bukan fatal, tapi bikin bobot belajar timpang ke pola tertentu.

In [5]:
from src.eval.data_quality_stats import dup_examples
rows = _read_jsonl(TRAIN['cot']); soals = _soals(rows, 'chatml')
ex = dup_examples(soals, thr=0.92, k=4)
print(f'cot: {len(ex)} pasang near-duplikat teratas (thr=0.92)')
for cos, a, b in ex:
    print('='*70); print(f'  cosine={cos}')
    print(f'    A: {a[:110]}')
    print(f'    B: {b[:110]}')

cot: 4 pasang near-duplikat teratas (thr=0.92)
  cosine=1.0
    A: Berapa banyak mol MgO yang dibutuhkan untuk bereaksi dengan 3 mol CO2 untuk membentuk 3 mol MgCO3
    B: Berapa banyak mol CO2 yang dibutuhkan untuk bereaksi dengan 3 mol MgO untuk membentuk 3 mol MgCO3
  cosine=1.0
    A: Berapa banyak mol HNO3 yang dibutuhkan untuk bereaksi dengan 2 mol NH3 untuk membentuk 2 mol NH4NO3
    B: Berapa banyak mol NH3 yang dibutuhkan untuk bereaksi dengan 2 mol HNO3 untuk membentuk 2 mol NH4NO3
  cosine=0.997
    A: Sekelompok 4 orang memutuskan untuk melepas sepatu mereka di luar perpustakaan untuk menghindari suara langkah
    B: Sekelompok 10 orang memutuskan untuk melepas sepatu mereka di luar perpustakaan untuk menghindari suara langka
  cosine=0.997
    A: Berapa banyak mol Hidrogen yang dibutuhkan untuk bereaksi dengan 1 mol Karbon untuk membentuk 1 mol Metana
    B: Berapa banyak mol karbon yang dibutuhkan untuk bereaksi dengan 2 mol hidrogen untuk membentuk 1 mol metana


## Bukti 3 - Soal yang TAK terdeteksi Bahasa Indonesia

Bisa = soal asing/sampah, ATAU positif-palsu (soal sangat pendek / dominan rumus matematika
yang bikin langdetect ragu). Eyeball buat bedakan.

In [6]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
rows = _read_jsonl(TEST['easy_test']); soals = _soals(rows, 'gold')
non_id = []
for s in soals:
    if len(s) < 20: continue
    try:
        if detect(s) != 'id': non_id.append(s)
    except Exception: pass
print(f'easy_test: {len(non_id)} soal non-id (dari yg cukup panjang)')
for s in non_id[:6]:
    print('  -', s[:100])

easy_test: 32 soal non-id (dari yg cukup panjang)
  - Jika $f(\theta) = (\frac{\cos \theta + \sin \theta}{\cos \theta - \sin \theta})^2$, maka hasil $f'(0
  - Hitunglah: $\int_{0}^{3} t\sqrt{t+1}dt$
  - Proyeksi ortogonal vektor $\vec{a}=4\vec{i}+\vec{j}+3\vec{k}$ pada $\vec{b}=2\vec{i}+\vec{j}+3\vec{k
  - Nilai dari $\frac{1}{2} + \frac{1}{6} + \frac{1}{12} + \frac{1}{20} + \dots + \frac{1}{10100}$ adala
  - jari-jari bola setelah ditiup adalah ...\nA. $\frac{1}{\sqrt{\pi}}$ cm\nB. $\frac{1}{\sqrt{2\pi}}$ c
  - Jika $f(x) = x^2 - 4x$, tentukan $f(x - 3)$.


## Kesimpulan

- **Kontaminasi:** `numglue_test` ~7% soal mirip train (cosine>=0.85), `easy_test` ~1%.
  Artinya skor eval numglue sedikit optimistik (sebagian soal mungkin "kenal" dari train);
  easy praktis bersih. Ini cek yang TAK ada di dua metode lain.
- **Label sehat:** `entropi_jawaban` tinggi (~90%+) & `top_jawaban_share` rendah (<10%) di semua
  set -> jawaban nyebar, BUKAN label degenerate. Satu mode kegagalan eval tersingkir.
- **Bahasa:** mayoritas terdeteksi Indonesia; `lang_id_rate` easy lebih rendah krn banyak soal
  pendek/penuh rumus (positif-palsu), bukan beneran asing -> lihat Bukti 3.
- **Near-duplikat train** ~5% -> redundansi ringan, bukan kerusakan.

**Gabungan tiga metode:**
- heuristik + judge sepakat **numglue gold banyak yang rusak** (label noise + butuh data luar).
- statistik nambah: **numglue juga sedikit terkontaminasi train** -> dobel alasan skor numglue tak bisa dipercaya apa adanya.
- train (cot/nocot) konsisten bersih di ketiga metode.

### Caveat
- TF-IDF char n-gram nangkep kemiripan permukaan (leksikal). Parafrase makna-sama tapi kata beda
  bisa lolos. Untuk lebih semantik: ganti ke embedding (sentence-transformers) -- lebih berat & nondeterministik.
- `lang_id_rate` & outlier = sinyal triase, bukan vonis. Konfirmasi dengan eyeball (sel Bukti).